In [9]:
import pandas as pd

# 1. Carrega os dados que agora já contém beta e momentum
df = pd.read_parquet("../../data/02_clean/df_metrics.parquet")
df["hrm"] = - df["hrm"]

# Labels padronizados dos decis
decile_labels = [f"decil_{i}" for i in range(1, 11)]

# 2. Função auxiliar para dividir em 10 decis
def create_deciles(series):
    return pd.qcut(series, q=10, labels=decile_labels, duplicates='drop')

# Aplica a criação de decis agrupando por ano (para que o rank seja justo dentro de cada ano)
df['decil_hrm'] = df.groupby('year')['hrm'].transform(create_deciles)
df['decil_pozzi'] = df.groupby('year')['pozzi'].transform(create_deciles)

# 3. Calcula o Beta e Momentum dos portfólios baseados em HRM
portfolios_hrm = df.groupby(['year', 'decil_hrm'], observed=False)[['beta', 'momentum']].mean().reset_index()
portfolios_hrm = portfolios_hrm.rename(columns={'decil_hrm': 'decil'})
portfolios_hrm['metric'] = 'hrm'

# 4. Calcula o Beta e Momentum dos portfólios baseados em Pozzi
portfolios_pozzi = df.groupby(['year', 'decil_pozzi'], observed=False)[['beta', 'momentum']].mean().reset_index()
portfolios_pozzi = portfolios_pozzi.rename(columns={'decil_pozzi': 'decil'})
portfolios_pozzi['metric'] = 'pozzi'

# 5. Junta tudo num único DataFrame limpo e organizado
df_portfolios = pd.concat([portfolios_hrm, portfolios_pozzi], ignore_index=True)

# Visualizando o resultado
display(df_portfolios.head(15))

,year,decil,beta,momentum,metric
0,2014,decil_1,1.145954,0.049056,hrm
1,2014,decil_2,1.113086,0.081365,hrm
2,2014,decil_3,1.104231,0.113841,hrm
3,2014,decil_4,1.021272,0.134645,hrm
4,2014,decil_5,0.870238,0.088924,hrm
5,2014,decil_6,0.817143,0.076725,hrm
6,2014,decil_7,0.741907,0.133376,hrm
7,2014,decil_8,0.740792,0.067853,hrm
8,2014,decil_9,0.575826,0.152218,hrm
9,2014,decil_10,0.134745,0.106006,hrm


In [10]:
pd.read_parquet("../../data/02_clean/df_metrics.parquet")

,node,hrm,pozzi,degree,closeness,eig,year,beta,momentum
0,A,-0.113502,0.162029,2.219453,0.289924,0.000978,2014,1.289674,0.036170
1,AA,-0.242146,0.222550,1.851429,0.271225,0.000676,2014,1.470522,0.518100
2,AAL,-0.411204,0.213583,2.391266,0.237571,0.000902,2014,1.605365,1.117107
3,AAME,-0.566156,0.212748,0.223170,0.226859,0.000381,2014,0.045365,-0.002096
4,AAON,0.892336,0.186629,2.481132,0.331172,0.034447,2014,1.682752,0.066075
...,...,...,...,...,...,...,...,...,...
40657,ZTS,-0.001498,0.140701,0.999036,0.266343,0.002999,2024,0.623014,-0.166471
40658,ZUMZ,0.723058,0.207884,3.742803,0.292672,0.016824,2024,1.244622,-0.073231
40659,ZVRA,-0.196636,0.189373,1.037889,0.242260,0.002037,2024,NaN,NaN
40660,ZWS,0.349889,0.202001,1.375150,0.285686,0.009684,2024,NaN,NaN


In [2]:
df_results

,Year,k,Centrality,Beta
0,2015,1,central,1.122735
1,2015,1,peripheral,0.578857
2,2015,10,central,0.980891
3,2015,10,peripheral,0.639678
4,2015,30,central,1.039241
5,2015,30,peripheral,0.624696
6,2016,1,central,1.547606
7,2016,1,peripheral,0.465895
8,2016,10,central,1.148203
9,2016,10,peripheral,0.702152


In [4]:
df_results.to_parquet("../../data/07_portfolios_metadata/beta_df.parquet")

In [1]:
import pandas as pd

years = range(2015, 2024)
returns_dict = {}

for year in years:
    for centrality in ["central", "peripheral"]:
        for k in [1, 10, 30]:
            returns_dict[f"{centrality}_{year}_{k}"] = pd.read_csv(f"../../data/06_portfolios/{centrality}_{year}_{k}.csv", index_col="Date")

In [2]:
import yfinance as yf

momentum_dict = {}

for portfolio_name, df_returns in returns_dict.items():
    first_price = yf.download(
        tickers=list(returns_dict[portfolio_name].columns),
        start=f"{year-1}-12-29",
        end=f"{year}-01-01"
    )["Close"]

    last_price = yf.download(
        tickers=list(returns_dict[portfolio_name].columns),
        start=f"{year}-12-29",
        end=f"{year+1}-01-01"
    )["Close"]

    momentum_dict[portfolio_name] = last_price.iloc[0] / first_price.iloc[0] - 1

[*********************100%***********************]  80 of 80 completed
[*********************100%***********************]  80 of 80 completed
[*********************100%***********************]  80 of 80 completed
[*********************100%***********************]  80 of 80 completed
[*********************100%***********************]  80 of 80 completed
[*********************100%***********************]  80 of 80 completed
[*********************100%***********************]  80 of 80 completed
[*********************100%***********************]  80 of 80 completed
[*********************100%***********************]  80 of 80 completed
[*********************100%***********************]  80 of 80 completed
[*********************100%***********************]  80 of 80 completed
[*********************100%***********************]  80 of 80 completed
[*********************100%***********************]  80 of 80 completed
[*********************100%***********************]  80 of 80 completed
[*****

KeyboardInterrupt: 

In [ ]:
momentum_df = (
    pd.concat(momentum_dict, names=["portfolio", "Ticker"])
    .reset_index(name=f"momentum12mo_{year}")
)
momentum_df.to_parquet(f"../../data/07_portfolios_metadata/momentum_df_{year}.parquet", index=False)

In [30]:
momentum_df

,portfolio,Ticker,momentum12mo_2024
0,central,AAPL,0.316343
1,central,ACWI,0.177692
2,central,ADI,0.088667
3,central,AMAT,0.017571
4,central,AMKR,-0.204909
...,...,...,...
138,peripheral,VIRC,-0.144975
139,peripheral,WFCF,-0.087085
140,peripheral,WKSP,-0.328859
141,peripheral,XOMA,0.410270


## Completo beta e momentum

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import statsmodels.api as sm
from tqdm import tqdm

df_metrics = pd.read_parquet("../../data/02_clean/df_metrics.parquet")
tickers = df_metrics['node'].unique().tolist()
years = df_metrics['year'].unique().tolist()

def beta_ols(rp, rm):
    rm = rm.squeeze().rename("rm")
    joined = rp.to_frame("rp").join(rm, how="inner").dropna()
    if len(joined) < 20:  # not enough observations
        return np.nan
    y = joined["rp"]
    X = sm.add_constant(joined["rm"])
    model = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
    return model.params["rm"]

def momentum_12m(prices):
    """Total return over the year."""
    prices = prices.dropna()
    if len(prices) < 2:
        return np.nan
    return (prices.iloc[-1] / prices.iloc[0]) - 1

records = []

for year in sorted(years):
    print(f"\n=== {year} ===")
    start, end = f"{year}-01-01", f"{year}-12-31"

    # Market returns
    sp500_prices = yf.download("^GSPC", start=start, end=end, progress=False)["Close"]
    sp500_dr = sp500_prices.pct_change().dropna()
    sp500_dr.name = "rm"

    # Tickers to process this year
    tickers_year = df_metrics[df_metrics['year'] == year]['node'].tolist()
    
    # Download all tickers at once
    raw = yf.download(tickers_year, start=start, end=end, progress=False)["Close"]
    if isinstance(raw, pd.Series):
        raw = raw.to_frame(name=tickers_year[0])

    for ticker in tqdm(tickers_year, desc=f"Computing {year}"):
        if ticker not in raw.columns:
            records.append({"node": ticker, "year": year, "beta": np.nan, "momentum": np.nan})
            continue
            
        prices = raw[ticker].dropna()
        returns = prices.pct_change().dropna()

        b = beta_ols(returns, sp500_dr)
        m = momentum_12m(prices)

        records.append({
            "node": ticker,
            "year": year,
            "beta": b,
            "momentum": m
        })

beta_momentum_df = pd.DataFrame(records)
print(beta_momentum_df.describe())


In [ ]:
# Join with df_metrics
df_metrics = df_metrics.merge(beta_momentum_df, on=["node", "year"], how="left")

# Save updated metrics
df_metrics.to_parquet("../../data/02_clean/df_metrics.parquet", index=False)
df_metrics.head()


In [15]:
metadata = pd.read_csv("../../data/02_clean/METADADOS_ATUALIZADO - Sheet1.csv")
metadata["mcap_2024"]

0      3.754818e+12
1      7.254202e+11
2      1.468970e+15
3      6.937964e+11
4      6.758384e+11
           ...     
394    2.054726e+09
395    5.491550e+09
396    5.117357e+10
397    1.722747e+11
398    1.385571e+11
Name: mcap_2024, Length: 399, dtype: float64